In [24]:
from pyspark.sql import SparkSession
spark= SparkSession.builder.appName("Missing").getOrCreate()

In [2]:
spark

In [25]:
#Read the data
training= spark.read.csv("./datasets/test1.csv", header= True, inferSchema= True)
training.show()

+---------+---+----------+------+
|     Name|age|Experience|Salary|
+---------+---+----------+------+
|    Krish| 31|        10| 30000|
|Sudhanshu| 30|         8| 25000|
|    Sunny| 29|         4| 20000|
|     Paul| 24|         3| 20000|
|   Harsha| 21|         1| 15000|
|  Shubham| 23|         2| 18000|
+---------+---+----------+------+



In [27]:
training.printSchema()

root
 |-- Name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- Experience: integer (nullable = true)
 |-- Salary: integer (nullable = true)



In [28]:
training.columns

['Name', 'age', 'Experience', 'Salary']

In [30]:

from pyspark.ml.feature import VectorAssembler    #By using VectorAssembler we can convert Input/Features into one vector 

featureassembler = VectorAssembler(
    inputCols=["age", "Experience"],
    outputCol="Independent Features"    #Naming convenction you can use Features simply
)

In [31]:
output= featureassembler.transform(training)  #Basically what we are saying is featureassembler---> we have instructions and transform(training) -->apply those to this dataset

In [32]:
output.show()

+---------+---+----------+------+--------------------+
|     Name|age|Experience|Salary|Independent Features|
+---------+---+----------+------+--------------------+
|    Krish| 31|        10| 30000|         [31.0,10.0]|
|Sudhanshu| 30|         8| 25000|          [30.0,8.0]|
|    Sunny| 29|         4| 20000|          [29.0,4.0]|
|     Paul| 24|         3| 20000|          [24.0,3.0]|
|   Harsha| 21|         1| 15000|          [21.0,1.0]|
|  Shubham| 23|         2| 18000|          [23.0,2.0]|
+---------+---+----------+------+--------------------+



In [33]:
output.columns

['Name', 'age', 'Experience', 'Salary', 'Independent Features']

In [34]:
#we want only two columns ["Salary", "Independent Features"]

finalized_data= output.select('Salary', 'Independent Features')
finalized_data.show()

+------+--------------------+
|Salary|Independent Features|
+------+--------------------+
| 30000|         [31.0,10.0]|
| 25000|          [30.0,8.0]|
| 20000|          [29.0,4.0]|
| 20000|          [24.0,3.0]|
| 15000|          [21.0,1.0]|
| 18000|          [23.0,2.0]|
+------+--------------------+



In [35]:
#Now we test wit LinearRegression

from pyspark.ml.regression import LinearRegression    #Importing LinearRegression model

In [36]:
#train-test split
train_data, test_data= finalized_data.randomSplit([0.75, 0.25])    #Train - Test method on data (75 train data and 25 data will be tested on Trained data) and spliting data Randomly

In [37]:
regressor= LinearRegression(
    featuresCol= "Independent Features", 
    labelCol= "Salary"
)

In [38]:
regressor= regressor.fit(train_data)      #we usually what we do is model.fit(X, y)   -- but here we already converted X and y to a vector and to a seperate column

In [39]:
#Checking what our model learned

# Coefficients
regressor.coefficients

DenseVector([28.4757, 1271.3568])

In [40]:
#Intercepts
regressor.intercept

14299.832495812996

In [41]:
#Prediction
pred_result= regressor.evaluate(test_data)

In [42]:
pred_result.predictions.show()

+------+--------------------+------------------+
|Salary|Independent Features|        prediction|
+------+--------------------+------------------+
| 30000|         [31.0,10.0]|27896.147403685147|
+------+--------------------+------------------+



In [45]:
#These are evaluation metrics for your regression model.  "How wrong is my model?"
pred_result.meanAbsoluteError, pred_result.meanSquaredError, pred_result.rootMeanSquaredError

(2103.852596314853, 4426195.747020748, 2103.852596314853)